# Colab Training Runbook

Launcher for one (model, dataset_size) training run on Colab GPU. The data-scaling study sweeps both `MODEL` and `N_TRAIN` across separate runs.

## Before Running

1. Select a GPU runtime: `Runtime > Change runtime type > GPU`.
2. Put `train.h5`, `val.h5`, and `test.h5` under `MyDrive/masters-thesis/data/scaling/`. The `train.h5` should contain at least `N_TRAIN` trajectories (use 10000 to cover all sizes).
3. Paste a GitHub token in Step 1 if the repo is private.

## Picking the Run

| Param | Values |
|---|---|
| `MODEL` | `hgnn` or `egnn` |
| `N_TRAIN` | 1000 / 2000 / 5000 / 10000 (slice of the train file) |
| `EPOCHS` | 300 default; both models peaked well before 300 in earlier runs |

## Persistent Outputs

Each run lives under its own (model, size) subtree:

```text
MyDrive/masters-thesis/runs/<model>/n<N_TRAIN>/<run_id>/
```

If Colab disconnects, the latest checkpoint and `metrics.csv` remain in Drive.


In [ ]:
# @title 1. Setup: Drive, model, and repo
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

GITHUB_TOKEN = ""  # @param {type:"string"}
REPO = "AlexNeagu123/msc-thesis-gnn-nbody"
DRIVE = Path("/content/drive/MyDrive/masters-thesis")

MODEL = "hgnn"  # @param ["hgnn", "egnn"]
N_TRAIN = "1000"  # @param ["1000", "2000", "5000", "10000"]
EPOCHS = 300  # @param {type:"integer"}
RUN_TRAINING = True  # @param {type:"boolean"}
RUN_EVALUATION = True  # @param {type:"boolean"}

N_TRAIN = int(N_TRAIN)
assert MODEL in {"egnn", "hgnn"}, MODEL
assert EPOCHS > 0, EPOCHS
assert N_TRAIN in (1000, 2000, 5000, 10000), N_TRAIN
if not GITHUB_TOKEN:
    raise ValueError("Paste a GitHub token before cloning the repo.")

repo_dir = Path("/content/repo")
clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"

get_ipython().run_cell_magic(
    "bash",
    "",
    f"""
set -e
if [ -d "{repo_dir}/.git" ]; then
  git -C "{repo_dir}" pull --ff-only
elif [ -e "{repo_dir}" ]; then
  echo "{repo_dir} exists but is not a git checkout" >&2
  exit 1
else
  git clone --depth 1 "{clone_url}" "{repo_dir}"
fi
""",
)

%cd /content/repo/impl


In [ ]:
# @title 2. Install dependencies and verify GPU
get_ipython().run_cell_magic(
    "bash",
    "",
    """
set -e
pip install -q -r requirements.txt
""",
)

import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Select a GPU runtime before continuing.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# @title 3. Copy scaling dataset from Drive and verify shapes
import h5py
import shutil

data_dir = Path("data/output/scaling")
data_dir.mkdir(parents=True, exist_ok=True)

for name in ["train.h5", "val.h5", "test.h5"]:
    src = DRIVE / "data" / "scaling" / name
    dst = data_dir / name
    if not src.exists():
        raise FileNotFoundError(f"Missing Drive data file: {src}")
    shutil.copy2(src, dst)

for name in ["train.h5", "val.h5", "test.h5"]:
    with h5py.File(data_dir / name, "r") as f:
        shape = f["trajectories"].shape
        print(f"{name}: {shape}")
        if name == "train.h5" and shape[0] < N_TRAIN:
            raise ValueError(f"train.h5 has {shape[0]} trajectories, but N_TRAIN={N_TRAIN}")


In [ ]:
# @title 4. Prepare config and persistent Drive paths
import yaml

cfg_path = Path(f"configs/{MODEL}.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["training"]["epochs"] = int(EPOCHS)
cfg["training"]["device"] = "auto"

run_root = DRIVE / "runs" / MODEL / f"n{N_TRAIN}"
run_root.mkdir(parents=True, exist_ok=True)
cfg["checkpointing"]["enabled"] = True
cfg["checkpointing"]["dir"] = str(run_root)
cfg["logging"]["enabled"] = True
cfg["logging"]["dir"] = str(run_root)

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

print(cfg_path)
print(cfg_path.read_text())


In [ ]:
# @title 5. Train selected model
run_root = DRIVE / "runs" / MODEL / f"n{N_TRAIN}"
run_root.mkdir(parents=True, exist_ok=True)

if RUN_TRAINING:
    print(f"Training {MODEL} at N_TRAIN={N_TRAIN}; outputs to {run_root}", flush=True)
    get_ipython().run_cell_magic(
        "bash",
        "",
        f"""
set -e
PYTHONUNBUFFERED=1 python -u -m training.train --config configs/{MODEL}.yaml --n-train {N_TRAIN}
""",
    )
else:
    print("Training skipped.")

run_ids = sorted(p.name for p in run_root.iterdir() if p.is_dir() and (p / "latest.pt").exists())
if not run_ids:
    raise RuntimeError(f"No checkpoint runs found under {run_root}.")

RUN_ID = run_ids[-1]
RUN_DIR = run_root / RUN_ID
BEST_CHECKPOINT = RUN_DIR / "best.pt"
LATEST_CHECKPOINT = RUN_DIR / "latest.pt"
print(f"Run id: {RUN_ID}")
print(f"Run directory: {RUN_DIR}")
print(f"Best checkpoint: {BEST_CHECKPOINT}")


In [ ]:
# @title 6. Run official numeric evaluation
EVAL_DIR = RUN_DIR / "evaluation"

if RUN_EVALUATION:
    get_ipython().run_cell_magic(
        "bash",
        "",
        f"""
set -e
PYTHONUNBUFFERED=1 python -u -m evaluation.evaluate \
  --config configs/{MODEL}.yaml \
  --checkpoint "{BEST_CHECKPOINT}" \
  --test-path data/output/scaling/test.h5 \
  --output-dir "{EVAL_DIR}" \
  --device cuda
""",
    )
else:
    print("Evaluation skipped.")

print(f"Evaluation directory: {EVAL_DIR}")
if EVAL_DIR.exists():
    print((EVAL_DIR / "summary.csv").read_text())


In [ ]:
# @title 7. Verify persistent Drive artifacts
items = [
    RUN_DIR / "best.pt",
    RUN_DIR / "latest.pt",
    RUN_DIR / "metrics.csv",
    RUN_DIR / "diagnostics.log",
    EVAL_DIR / "summary.csv",
    EVAL_DIR / "metrics.json",
]

for path in items:
    status = "ok" if path.exists() else "missing"
    print(f"{status}: {path}")

print(f"Drive output: {RUN_DIR}")